# My First Dummy AI Agent — Clean Notebook

This notebook builds the Hugging Face Unit 1 **Thought → Action → Observation** demo using:
- Python
- Google Gemini API through Google's OpenAI-compatible endpoint




## 1. Install the client library

We use the `openai` Python package because Google's Gemini API supports an OpenAI-compatible endpoint. We are **not** sending requests to Hugging Face in this notebook.

In [31]:
%pip install -q -U openai

## 2. Create one Gemini client

Keep one client variable named `gemini_client` throughout the notebook. Do not overwrite it with a Hugging Face `InferenceClient`.

In [46]:
from google.colab import userdata
from openai import OpenAI

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY not found. Add it in Colab Secrets and enable notebook access."
    )

MODEL_NAME = "gemini-3.6-flash"

gemini_client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

print("Gemini client ready.")
print("Model:", MODEL_NAME)
print("Endpoint:", gemini_client.base_url)

Gemini client ready.
Model: gemini-3.6-flash
Endpoint: https://generativelanguage.googleapis.com/v1beta/openai/


## 3. Test a simple LLM request

This is only a connection test. It is not an Agent yet.

In [50]:
test_response = gemini_client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "Reply with exactly: OK"}
    ],
    max_tokens=256
)

test_message = test_response.choices[0].message
print("Finish reason:", test_response.choices[0].finish_reason)
print("Model reply:", test_message.content if test_message.content else test_message)

Finish reason: stop
Model reply: OK


## 4. Define the Agent's system prompt

The system prompt tells the model which tool exists and what JSON action format to produce. It also tells the model not to invent an Observation: Python will execute the tool and add the real tool result.

In [34]:
SYSTEM_PROMPT = """
You are a simple weather agent.

You have exactly one available tool:
- get_weather(location): returns a weather report for the requested location.

When the user asks for weather:
1. Briefly state the task in a Thought line.
2. Return exactly one Action as a JSON object with:
   - "action": "get_weather"
   - "action_input": {"location": "<city>"}
3. Do not invent or write an Observation. The Python runtime will execute the tool
   and provide the real tool result.

Use this format:

Thought: I need to check the weather for the requested location.
Action:
```json
{
  "action": "get_weather",
  "action_input": {
    "location": "London"
  }
}
```

After an Observation is supplied in the conversation, use that Observation
to answer the original question. Do not claim that the dummy tool provides
live weather.
"""

print("System prompt created.")

System prompt created.


## 5. Prepare the user's question and messages

The `system` message contains Agent instructions. The `user` message contains the task.

In [48]:
user_question = "What's the weather in London?"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_question}
]

print(messages)

[{'role': 'system', 'content': '\nYou are a simple weather agent.\n\nYou have exactly one available tool:\n- get_weather(location): returns a weather report for the requested location.\n\nWhen the user asks for weather:\n1. Briefly state the task in a Thought line.\n2. Return exactly one Action as a JSON object with:\n   - "action": "get_weather"\n   - "action_input": {"location": "<city>"}\n3. Do not invent or write an Observation. The Python runtime will execute the tool\n   and provide the real tool result.\n\nUse this format:\n\nThought: I need to check the weather for the requested location.\nAction:\n```json\n{\n  "action": "get_weather",\n  "action_input": {\n    "location": "London"\n  }\n}\n```\n\nAfter an Observation is supplied in the conversation, use that Observation\nto answer the original question. Do not claim that the dummy tool provides\nlive weather.\n'}, {'role': 'user', 'content': "What's the weather in London?"}]


## 6. Ask the LLM for an Action, and stop before Observation

`stop=["Observation:"]` asks the model generation to stop at that marker. The model should return a Thought and JSON Action, not make up the tool result.

In [49]:
action_response = gemini_client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages, #mundhu create chesina model ikada peduthunnam
    max_tokens=500,
    stop=["Observation:"] #to stop hallucination we stop here
)

agent_response = action_response.choices[0].message.content #choices are model response choice and [0] first choice abd message object access chestunnam and .content vachesi actual generated text we are extract

if not agent_response:
    raise RuntimeError(
        "The model returned no text content. "
        f"Finish reason: {action_response.choices[0].finish_reason}. "
        "Inspect action_response.choices[0].message for details."
    )

print(agent_response)

Thought: I need to check the weather for London.
Action:
```json
{
  "action": "get_weather",
  "action_input": {
    "location": "London"
  }
}
```


## 7. Parse the JSON Action

The parser extracts JSON text and converts it to a Python dictionary. Then we read the action name and arguments.

In [51]:
import json #json data ni we convert in to python dict
import re #Text lo patterns search cheyadaniki and for identify

# First try to find a fenced JSON block.
match = re.search( #text lo particular pattrn ni search chestundi
    r"```(?:json)?\s*(.*?)\s*```", #Regex pattrn '''openining code block and optional ga json label undchu and \s spaces and new lines allow chestundi and Json context ni cpature chesthundi and closing spaces mundhu allow chesthundi and closing the code block
    agent_response,
    re.DOTALL | re.IGNORECASE #multiple line lo unna JSON capture and again and after the next line vachesi lower and upper kakunda it will take edhanna match avaachu
)

if match:
    json_text = match.group(1) #group 1 ante regex lo first capturing group like (.*?)
    ''' example:
{
    "action": "get_weather",
    "action_input": {
        "location": "London"
    }
}

    '''
else:
    # Fallback: extract the first JSON object from the response.
    #
    start = agent_response.find("{") #first occurence find()
    end = agent_response.rfind("}") #last occurence rfind()
    if start == -1 or end == -1 or end < start:
        raise ValueError(f"No JSON action found in model response:\n{agent_response}")
    json_text = agent_response[start:end + 1] #extract text from opening to closing brackets

action_data = json.loads(json_text)
'''
json string load in to python dic
json_text = """
{
    "action": "get_weather",
    "action_input": {
        "location": "London"
    }
}
"""
like this :
action_data = {
    "action": "get_weather",
    "action_input": {
        "location": "London"
    }
}
 '''

action_name = action_data.get("action") #dict nunchi extract the two values like get_weather okati action lo undhi
action_input = action_data.get("action_input") #.get use chesi andhulo unna value

if not isinstance(action_input, dict):
    '''
    isinstance() oka value particular data type ki belong avuthunda ani check chestundi
    like this:
    {
    "location": "London"
}
    '''
    raise ValueError("action_input must be a JSON object.")

print("Action:", action_name)
print("Action input:", action_input)

Action: get_weather
Action input: {'location': 'London'}


In [54]:
import requests

def get_weather(location: str) -> str:

    # 1. Find city coordinates
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"

    geo_response = requests.get(
        geo_url,
        params={
            "name": location,
            "count": 1,
            "language": "en",
            "format": "json"
        },
        timeout=15
    )

    geo_response.raise_for_status()

    places = geo_response.json().get("results", [])

    if not places:
        raise ValueError(f"Location not found: {location}")

    place = places[0]

    latitude = place["latitude"]
    longitude = place["longitude"]

    # 2. Fetch weather information
    weather_url = "https://api.open-meteo.com/v1/forecast"

    weather_response = requests.get(
        weather_url,
        params={
            "latitude": latitude,
            "longitude": longitude,
            "current": "temperature_2m,relative_humidity_2m,weather_code,wind_speed_10m",
            "timezone": "auto"
        },
        timeout=15
    )

    weather_response.raise_for_status()

    current = weather_response.json()["current"]

    weather_codes = {
        0: "Clear sky",
        1: "Mainly clear",
        2: "Partly cloudy",
        3: "Overcast",
        45: "Fog",
        48: "Fog",
        51: "Light drizzle",
        53: "Drizzle",
        55: "Heavy drizzle",
        61: "Light rain",
        63: "Moderate rain",
        65: "Heavy rain",
        71: "Light snow",
        73: "Snow",
        75: "Heavy snow",
        80: "Rain showers",
        81: "Rain showers",
        82: "Heavy rain showers",
        95: "Thunderstorm"
    }

    condition = weather_codes.get(
        current["weather_code"],
        "Condition unavailable"
    )

    return (
        f"Weather in {place['name']}, {place.get('country', '')}: "
        f"{condition}. "
        f"Temperature: {current['temperature_2m']}°C. "
        f"Humidity: {current['relative_humidity_2m']}%. "
        f"Wind speed: {current['wind_speed_10m']} km/h."
    )

print("Real weather tool created successfully!")

Real weather tool created successfully!


## 9. Execute the selected tool and capture the Observation

The runtime checks the tool allowlist before calling the function. This is a basic example of validating the requested action.

In [41]:
tools = {
    "get_weather": get_weather
}

if action_name not in tools:
    raise ValueError(f"Unknown or disallowed action: {action_name}")
#exact ga location argument undali ani check chesthunnam
if set(action_input) != {"location"}:
    raise ValueError("get_weather expects exactly one argument: location")

location = action_input["location"]
#ee lines lo two validation like location lo unna spaces remove by strip
if not isinstance(location, str) or not location.strip():
    raise ValueError("location must be a non-empty string")
#ikada actual info like get_weather and the location is london
observation = tools[action_name](location=location)

print("Observation:", observation)

Observation: Weather in London, United Kingdom: Overcast. Temperature: 9.0°C. Humidity: 77%. Wind speed: 12.6 km/h.


## 10. Append the Observation to the conversation context

We add the model's Action and the Python tool's Observation to the conversation. This updates the context for the next LLM call; it does not train the model or change its weights.

In [53]:
print("System Prompt:", bool(SYSTEM_PROMPT))
print("User Question:", bool(user_question))
print("Agent Response:", bool(agent_response))
print("Observation:", bool(observation))

System Prompt: True
User Question: True
Agent Response: True
Observation: True


In [39]:
final_messages = [ #indulo complete llm pampinchey untadi response
    #Mana Agent ki mundu define chesina instructions ikkada pampistunnam
     {
         "role": "system",
         "content": SYSTEM_PROMPT
     },
    #User actual ga adigina question ni add chestunnam
     {
         "role": "user",
         "content": user_question
     },
     # in below lines we are passing the agent response to the
     # two new lines create chesi observation anedhi seperate ga we are passing
     # in below \n\n andhukey two new lines
     {
         "role": "assistant",
         "content": agent_response + "\n\nObservation:\n" + observation #three strings here we are concatenate chestunnam
     },
     {
         #we are asking the final answer to the llm
         "role": "user",
         "content": "Now answer my original question using the Observation above."
     }
]

print("Updated context prepared.")
print(final_messages[-2]["content"]) #content lo unna 2nd last ni manam extract chesthunnam and msg lo unna action and observation kuda

Updated context prepared.
Thought: I need to check the weather for London.
Action:
```json
{
  "action": "get_weather",
  "action_input": {
    "location": "London"
  }
}
```

Observation:
The weather in London is sunny with low temperatures. (Dummy example; not live weather data.)


## 11. Ask the LLM for the final answer

The second LLM call receives the user's question, the Agent's Action, and the real result returned by the Python tool.

In [38]:
#second llm request here calling and 1st lo json action and 2nd observation read chesi final output
final_response = gemini_client.chat.completions.create(
    model=MODEL_NAME,
    messages=final_messages,
    #Ikkada mana previously create chesina final_messages ni Gemini ki send chestunnam.
    max_tokens=400
)

final_message = final_response.choices[0].message
final_answer = final_message.content #complete response no need and we want the actual text want to display to the user

if not final_answer:
    print("No text content returned. Message details:")
    print(final_message)
else:
    print(final_answer)

Based on the observation above, the weather in London is currently sunny with low temperatures (note: this is based on a dummy report and does not reflect live weather data).


## 12. What we built

The complete execution flow is:

**User question → LLM Thought/Action → Stop → Parse JSON → Execute Python tool → Observation → Append to context → Second LLM call → Final answer**

Try changing `user_question` to another city and run from Step 5 onward. Because the weather function is dummy, it will return the same sample weather description for every city.

In [37]:
print(get_weather("London"))

Weather in London, United Kingdom: Overcast. Temperature: 9.0°C. Humidity: 77%. Wind speed: 12.6 km/h.
